# Libraries

In [1]:
import torch
import random
import numpy as np

from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
from huggingface_hub import login
from huggingface_hub import create_repo, upload_file

# Global Variables

In [2]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_DIR = "/kaggle/working/model"
REPO_ID = "EdgeCompress01/Llama-3.2-3B-Instruct-GGUF"
model_name_in_repo = "Llama-3.2-3B-Instruct-Q5_K_M.gguf"
SEED = 42

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Set Seed

In [4]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [5]:
# --- 3. HUGGING FACE AUTHENTICATION ---
print("Logging into Hugging Face...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

Logging into Hugging Face...


# Load Model

In [6]:
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    allow_patterns=[
        "*.safetensors",
        "config.json",
        "tokenizer*",
        "generation_config.json",
        "special_tokens_map.json"
    ]
)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

'/kaggle/working/model'

**Clone llama.cpp**

In [7]:
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 83043, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 83043 (delta 87), reused 51 (delta 51), pack-reused 82898 (from 3)
Receiving objects: 100% (83043/83043), 312.15 MiB | 19.33 MiB/s, done.
Resolving deltas: 100% (59788/59788), done.
Updating files: 100% (2437/2437), done.
/kaggle/working/llama.cpp


**Build llama.cpp**

In [8]:
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

# Quantization

**Convert from huggingface to gguf**

In [9]:
!python convert_hf_to_gguf.py ../model \
    --outfile ../model/llama-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,     

**Free space**

In [10]:
!rm -rf ../model/*.safetensors
!rm -rf ../model/tokenizer*
!rm -rf ../model/config*

**Quantize**

In [11]:
!./build/bin/llama-quantize \
    ../model/llama-f16.gguf \
    ../model/llama-q5.gguf \
    q5_k_m

main: build = 8279 (4d99d4508)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '../model/llama-f16.gguf' to '../model/llama-q5.gguf' as Q5_K_M
llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from ../model/llama-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.900000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   4:                               general.name str              = Model
llama_model_loader: - kv   5:                         general.size_label str              = 3.2B
llama_model_loader:

**Free space**

In [12]:
!rm ../model/llama-f16.gguf

# PUSH TO HUGGING FACE

In [13]:
# Create repo
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# Upload quantized model
upload_file(
    path_or_fileobj=f"{MODEL_DIR}/llama-q5.gguf",
    path_in_repo=model_name_in_repo,
    repo_id=REPO_ID,
    repo_type="model"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!
